# 03 · Why a more complex model is usually a worse one

### Session 2 support notebook | Simulation

---

Everything so far has been about the work around the model. This notebook is about
the model itself, and it answers one question:

> **Given a choice between a simple model and a complex one, which should you
> reach for, and how would you know you were wrong?**

We use simulated data for a reason you will not get anywhere else: here we know the
true relationship. In every real project you are guessing at it. Watching models
succeed and fail against a known truth builds an intuition you can then carry into
problems where the truth is hidden.

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 4.5)

## 1. A world where we know the truth

The true relationship is `y = sin(x)`. Reality adds noise, which is everything the
model cannot possibly know. No model can beat the noise floor. A model that appears
to has memorised the noise, and it will fail on anything new.

In [ ]:
def simulate(n=60, seed=321):
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 2*np.pi, n)
    y = np.sin(x) + rng.normal(0, 0.5, n)
    return pd.DataFrame({"x": x, "y": y})

train = simulate(seed=321)
test  = simulate(seed=987)       # same process, different draw: the unseen data

plt.scatter(train.x, train.y, label="training sample", color="#0F766E", alpha=.7)
plt.plot(train.x, np.sin(train.x), color="black", lw=2, label="the truth: sin(x)")
plt.legend(); plt.title("What the model is trying to recover"); plt.tight_layout(); plt.show()

print(f"Noise floor (best achievable test MSE): {0.5**2:.3f}")

**Write that number down: 0.25.** It is the variance of the noise, and it is the
best test MSE any model can achieve on this problem. Any model reporting a *training*
error well below it is memorising.

## 2. Model zero: the mean

Before any algorithm, the baseline. Predict the average of `y` for every `x`.
Every model from here must beat this, or it has contributed nothing.

In [ ]:
models = {}      # name -> (predict_fn, description)

mean_value = train.y.mean()
models["Mean (baseline)"] = (lambda d: np.full(len(d), mean_value), "no inputs used at all")

plt.scatter(train.x, train.y, color="#94A3B8", alpha=.6)
plt.plot(train.x, [mean_value]*len(train), "k--", lw=2)
plt.title(f"The baseline: predict {mean_value:.3f} for everything")
plt.tight_layout(); plt.show()

## 3. Polynomials of rising complexity

In [ ]:
def poly_features(d, degree):
    """x, x^2, ... x^degree. Nothing else."""
    out = pd.DataFrame({"x": d.x})
    for p in range(2, degree + 1):
        out[f"x{p}"] = np.power(d.x, p)
    return out

def fit_polynomial(train_df, degree):
    m = LinearRegression().fit(poly_features(train_df, degree), train_df.y)
    return lambda d, m=m, degree=degree: m.predict(poly_features(d, degree))

for deg in [1, 2, 3, 5, 10, 15]:
    models[f"Polynomial degree {deg}"] = (fit_polynomial(train, deg), f"{deg} terms in x")

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, deg in zip(axes.ravel(), [1, 2, 3, 5, 10, 15]):
    ax.scatter(train.x, train.y, color="#94A3B8", alpha=.5, s=18)
    ax.plot(train.x, np.sin(train.x), color="black", lw=1.5, alpha=.6)
    ax.plot(train.x, models[f"Polynomial degree {deg}"][0](train), color="#0F766E", lw=2.2)
    ax.set_title(f"degree {deg}")
plt.tight_layout(); plt.show()

**READ and answer.** Degree 15 passes closer to more of the training points than
degree 3 does.

> *Which of the two would you rather deploy, and what evidence would settle it?*

The evidence is in section 6. Commit to an answer first.

## 4. Decision trees, constrained and unconstrained

In [ ]:
tree_specs = {
    "Tree (unconstrained)":   DecisionTreeRegressor(random_state=123),
    "Tree (max_depth=3)":     DecisionTreeRegressor(max_depth=3, random_state=123),
    "Tree (min_leaf=10)":     DecisionTreeRegressor(min_samples_leaf=10, random_state=123),
}
fitted_trees = {}
for name, est in tree_specs.items():
    est.fit(train[["x"]], train.y)
    fitted_trees[name] = est
    models[name] = (lambda d, e=est: e.predict(d[["x"]]), "single tree")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (name, est) in zip(axes, fitted_trees.items()):
    ax.scatter(train.x, train.y, color="#94A3B8", alpha=.5, s=18)
    ax.plot(train.x, np.sin(train.x), color="black", lw=1.5, alpha=.6)
    ax.plot(train.x, est.predict(train[["x"]]), color="#B45309", lw=2.2)
    ax.set_title(f"{name}\n{est.get_depth()} levels, {est.get_n_leaves()} leaves", fontsize=10)
plt.tight_layout(); plt.show()

The unconstrained tree has one leaf per training point. It reproduces the training
data exactly, including every bit of noise, and its prediction for any new `x` is
whatever the nearest memorised point happened to be.

### Looking inside the tree

`graphviz` is not reliably available in Colab and the original version of this
notebook failed here. `sklearn.tree.plot_tree` needs nothing extra.

In [ ]:
plt.figure(figsize=(14, 5))
plot_tree(fitted_trees["Tree (max_depth=3)"], feature_names=["x"],
          filled=True, rounded=True, fontsize=9, impurity=False)
plt.title("A depth-3 tree is a readable set of rules. An unconstrained one is not.")
plt.tight_layout(); plt.show()

## 5. Random forests

In [ ]:
for name, est in {
    "Forest (unconstrained)": RandomForestRegressor(random_state=123),
    "Forest (min_leaf=10)":   RandomForestRegressor(min_samples_leaf=10, random_state=123),
}.items():
    est.fit(train[["x"]], train.y)
    models[name] = (lambda d, e=est: e.predict(d[["x"]]), "ensemble of trees")

print(f"{len(models)} models fitted.")

---

# 6. The comparison table

This is the payoff. Every model, scored on training data it has seen and on test
data it has not, side by side.

In [ ]:
rows = []
for name, (predict, _) in models.items():
    tr_pred, te_pred = predict(train), predict(test)
    rows.append({
        "model": name,
        "train MSE": mean_squared_error(train.y, tr_pred),
        "test MSE":  mean_squared_error(test.y,  te_pred),
        "test MAE":  mean_absolute_error(test.y, te_pred),
    })

results = pd.DataFrame(rows).set_index("model")
results["overfit gap"] = results["test MSE"] - results["train MSE"]
results = results.sort_values("test MSE")

print(f"Noise floor: {0.25:.3f}\n")
print(results.round(3).to_string())

**READ the three columns together. Each one tells you something different.**

**`train MSE`** rewards memorisation. Ranked on this column alone the unconstrained
models win, which is exactly why no practitioner ever ranks on it.

**`test MSE`** is the honest score. Note where the unconstrained tree and the
unconstrained forest sit: near the bottom on training error and near the bottom of
the *table* on test error. They memorised.

**`overfit gap`** is the diagnosis. A large gap means the model learned noise. A
small gap with a poor test score means the opposite problem: the model is too rigid
to represent the real pattern. Underfitting and overfitting are both failures, and
the gap tells you which one you have.

## 7. The complexity curve

The table shows the result. This shows the mechanism, and it is the single most
useful diagnostic plot in supervised learning.

In [ ]:
degrees = range(1, 16)
tr_err, te_err = [], []
for deg in degrees:
    f = fit_polynomial(train, deg)
    tr_err.append(mean_squared_error(train.y, f(train)))
    te_err.append(mean_squared_error(test.y,  f(test)))

best = int(np.argmin(te_err)) + 1
plt.plot(list(degrees), tr_err, "o-", color="#94A3B8", lw=2, label="training error")
plt.plot(list(degrees), te_err, "o-", color="#0F766E", lw=2, label="test error")
plt.axhline(0.25, color="black", ls=":", label="noise floor")
plt.axvline(best, color="#B45309", ls="--", label=f"best degree = {best}")
plt.xlabel("polynomial degree (model complexity)"); plt.ylabel("mean squared error")
plt.title("Training error always falls. Test error does not.")
plt.xticks(list(degrees)[::2]); plt.legend(); plt.tight_layout(); plt.show()

print(f"Best test MSE {min(te_err):.3f} at degree {best}, against a noise floor of 0.25")

**Training error only ever falls.** Add complexity and it keeps dropping. This is why
training error is not evidence of anything.

**Test error falls, bottoms out, then drifts upward.** The turning point is where the
model stops learning the pattern and starts learning the noise. Everything to the left
is underfitting, everything to the right is overfitting.

The polynomial curve shows you the *shape* of the effect. For the *magnitude*, look
back at the trees in the comparison table: the unconstrained tree scores a perfect
0.000 on training data and 0.474 on test, while the same algorithm with
`min_samples_leaf=10` scores worse on training and far better on test. A single
constraint, and the model went from useless to usable.

In a real project you cannot see this curve, because you only get to use the test set
once. What you do instead is cross-validation on the training set, which estimates it.
Notebook 04 does exactly that.

### BUILD: does this hold for trees?

In [ ]:
# BUILD: reproduce the curve above using max_depth from 1 to 15 instead of
# polynomial degree. Then answer:
#   (a) does test error turn upward in the same way?
#   (b) where is the turning point, and what does that tell you about tree depth
#       as a complexity knob?

depths = range(1, 16)
# YOUR CODE HERE

In [ ]:
# Solution
tr_e, te_e = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=123).fit(train[["x"]], train.y)
    tr_e.append(mean_squared_error(train.y, m.predict(train[["x"]])))
    te_e.append(mean_squared_error(test.y,  m.predict(test[["x"]])))

b = int(np.argmin(te_e)) + 1
plt.plot(list(depths), tr_e, "o-", color="#94A3B8", lw=2, label="training error")
plt.plot(list(depths), te_e, "o-", color="#B45309", lw=2, label="test error")
plt.axhline(0.25, color="black", ls=":", label="noise floor")
plt.axvline(b, color="#0F766E", ls="--", label=f"best depth = {b}")
plt.xlabel("max_depth"); plt.ylabel("mean squared error")
plt.title("Same shape, different knob"); plt.legend(); plt.tight_layout(); plt.show()

### What to carry into notebook 04

1. **Training error is not evidence.** It falls with complexity no matter what.
2. **Every model is scored against a baseline**, and the baseline here used no inputs.
3. **The noise floor exists.** Some of the variation in any target is irreducible,
   and a model that appears to beat it is telling you about a leak, not a triumph.
4. **Constraints are not a limitation, they are the design.** `max_depth`,
   `min_samples_leaf` and regularisation strength are how you sit at the minimum of
   that curve rather than to the right of it.

**Next:** `04_Model_Training_and_Selection` applies all four to the churn case.